# Partitioned LDSC

Goal: run partitioned LDSC in the refactored codebase by building query annotations, computing baseline-plus-query LD scores from an R2-table reference panel, and regressing one trait on that partitioned design.

Path-token rules used below:
- use `@` for chromosome suites such as `baseline.@`
- use globs when the filenames do not follow the simple chromosome-suffix convention
- scalar inputs still resolve to exactly one file
- output prefixes remain literal destinations


# global config

In [1]:
import os
print(f"Current working directory is {os.getcwd()}")

Current working directory is /Users/wenbinwu/Documents_local/Research/SullivanLab/LDSC/repos/ldsc_py3_Jerry_workspace/ldsc_py3_restructured/tutorials


In [ ]:
from pathlib import Path

import sys
sys.path.append("/Users/wenbinwu/Documents_local/Research/SullivanLab/LDSC/repos/ldsc_py3_Jerry_workspace/ldsc_py3_restructured/src")

from ldsc import AnnotationBuildConfig, AnnotationBuilder, AnnotationSourceSpec, CommonConfig, LDScoreCalculator, LDScoreConfig, RefPanelLoader, RefPanelSpec, RegressionConfig, RegressionRunner, run_bed_to_annot

# configure common settings
DATA_DIR = Path("../../resources")
OUTPUT_DIR = Path("../../result_consistency_checks/output_new")
common = CommonConfig(snp_identifier="snpid", genome_build="hg19")


# Processing annotation files

In [3]:
# convert BED files to annotation files
run_bed_to_annot(
    bed_files=str(DATA_DIR / "annot_raw" / "*.bed"),
    baseline_annot_dir=DATA_DIR / "baseline_v1.2",
    output_dir=OUTPUT_DIR / "annot_processed",
)

INFO: Inferred canonical field 'POS' from input column 'BP' in ../../resources/baseline_v1.2/baseline.1.annot.


/Users/wenbinwu/Documents_local/Research/SullivanLab/LDSC/repos/ldsc_py3_Jerry_workspace/ldsc_py3_restructured/src/ldsc/_kernel/annotation.py:858: UserWarning: Inferred canonical field 'POS' from input column 'BP' in ../../resources/baseline_v1.2/baseline.1.annot.
  column = resolve_required_column(header, spec, context=str(path))
INFO: Inferred canonical field 'POS' from input column 'BP' in ../../resources/baseline_v1.2/baseline.1.annot.gz.
/Users/wenbinwu/Documents_local/Research/SullivanLab/LDSC/repos/ldsc_py3_Jerry_workspace/ldsc_py3_restructured/src/ldsc/_kernel/annotation.py:858: UserWarning: Inferred canonical field 'POS' from input column 'BP' in ../../resources/baseline_v1.2/baseline.1.annot.gz.
  column = resolve_required_column(header, spec, context=str(path))
INFO: Inferred canonical field 'POS' from input column 'BP' in ../../resources/baseline_v1.2/baseline.10.annot.gz.
/Users/wenbinwu/Documents_local/Research/SullivanLab/LDSC/repos/ldsc_py3_Jerry_workspace/ldsc_py3_rest

PosixPath('../../result_consistency_checks/output_new/annot_processed')

# Calculate ld scores

In [4]:
# create annotation bundle
annotation_bundle = AnnotationBuilder(common, AnnotationBuildConfig()).run(
    AnnotationSourceSpec(
        baseline_annot_paths=DATA_DIR / "baseline_v1.2" / "baseline.@",
        query_annot_paths=str(OUTPUT_DIR / "annot_processed" / "query.@"),
    )
)

In [5]:
# build ref panel from R2 table
ref_panel = RefPanelLoader(common).load(
    RefPanelSpec(
        backend="parquet_r2",
        r2_table_paths=str(DATA_DIR / "ref_panel" / "MAF0.01_SNPonly_hm3only" / "EUR_chr@_LD_MAF0.01_SNPonly_hm3only.parquet"),
        chromosomes=tuple(annotation_bundle.chromosomes),
    )
)

In [6]:
# calculate ld scores with the workflow-layer API
ldscore_result = LDScoreCalculator().run(
    annotation_bundle=annotation_bundle,
    ref_panel=ref_panel,
    ldscore_config=LDScoreConfig(ld_wind_cm=1.0),
    common_config=common,
)

runner = RegressionRunner(common, RegressionConfig())

Since the annotation files and reference panel do not cover exactly the same SNP universe, the LD-score run can only use SNPs that exist in both:
- the annotation bundle
- the R2 table

In [7]:
# compare the number of SNPs per chromosome before and after constructing ref_panel
before = annotation_bundle.metadata["CHR"].value_counts().sort_index()
after = ldscore_result.reference_metadata["CHR"].value_counts().sort_index()

comparison = (
    before.rename("annotation_bundle")
    .to_frame()
    .join(after.rename("retained_in_ldscore"), how="outer")
    .fillna(0)
    .astype(int)
)
comparison["dropped"] = comparison["annotation_bundle"] - comparison["retained_in_ldscore"]
comparison

,annotation_bundle,retained_in_ldscore,dropped
CHR,,,
1,779354,94671,684683
10,510501,61604,448897
11,493922,58718,435204
12,480110,56339,423771
13,366200,43796,322404
14,324698,37887,286811
15,287001,34367,252634
16,316981,34941,282040
17,269222,30586,238636


# PLDSC

In [11]:
# estimate partitioned heritability for each query
from ldsc import load_sumstats

sumstats = load_sumstats(
    str(OUTPUT_DIR / "sumstats_processed" / "mdd2025.sumstats.gz"),
    trait_name="mdd2025",
)

partitioned = runner.estimate_partitioned_h2_batch(
    sumstats,
    ldscore_result,
    annotation_bundle,
)

# save results
partitioned.to_csv(Path(OUTPUT_DIR / "pldsc_results" / "partitioned_h2.tsv"), sep="\t", index=False)
print(partitioned)

  query_annotation   coefficient  coefficient_se  coefficient_z   
0  Cerebellum_PC16  1.801722e-09    3.141039e-09       0.573607  \
1   Cerebellum_PP1 -1.161059e-09    3.352352e-09      -0.346342   
2   Cerebellum_PP3 -1.924114e-09    2.914703e-09      -0.660141   
3  Hippocampus_PP1  4.501279e-10    3.239450e-09       0.138952   

   coefficient_p  category_h2  category_h2_se  proportion_h2   
0       0.566234     0.000237        0.000413       0.005807  \
1       0.729086    -0.000161        0.000466      -0.003963   
2       0.509164    -0.000260        0.000394      -0.006383   
3       0.889488     0.000070        0.000505       0.001723   

   proportion_h2_se  enrichment  
0          0.010139    0.657861  
1          0.011453   -0.424499  
2          0.009664   -0.703261  
3          0.012418    0.164747  
